# Module 3 — Intent Classifier

TF-IDF + LinearSVC on the gold `intent` column of the Bitext dataset, collapsed from 27 fine intents into 7 coarse routing categories. Supervised on labeled data (not zero/few-shot) since the labels already exist — see `src/intent/train.py` docstring for the full rationale.

In [ ]:
import sys
sys.path.append('..')
from src.intent.train import load_data, build_pipeline
from src import config
import pandas as pd

## Load & inspect

In [ ]:
df = load_data()
df['coarse_intent'].value_counts()

In [ ]:
df[['text', 'intent', 'coarse_intent']].sample(10, random_state=0)

## Train / test split & fit

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['coarse_intent'], test_size=0.15, stratify=df['coarse_intent'], random_state=42
)
pipe = build_pipeline()
pipe.fit(X_train, y_train)

## Evaluate

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

preds = pipe.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax, xticks_rotation=45)
plt.title('Intent — confusion matrix')
plt.tight_layout()
plt.show()

## Save

In [ ]:
import joblib
joblib.dump(pipe, config.INTENT_MODEL_PATH)
print('Saved to', config.INTENT_MODEL_PATH)

## Spot-check

In [ ]:
from src.intent.predict import predict_intent
for s in ['Hi there!', 'Where is my order #4021?', 'I want a refund, this is unacceptable.',
          'How do I reset my password?', "What's the weather like today?"]:
    print(s, '->', predict_intent(s))